# 04 — NER with LLM (Qwen2.5-7B-Instruct)

Runs named entity recognition on the **199-article validation sample** using `Qwen/Qwen2.5-7B-Instruct`
via the HuggingFace Inference API (`together` provider).  

**Design rationale**: The pipeline systems (spaCy, Stanza, Flair) operate on the **original German** text.  
The LLM operates on **English translations** (`content_en`). This asymmetry is intentional — it reflects  
the real-world constraint that English-speaker researchers would use: LLMs prompting in the target  
language they understand. Translation quality is tracked via `translation_quality` and reported separately.

**Output**: Adds `ner_llm` and `llm_entity_count` columns to the pipeline results DataFrame,  
in the same schema as `ner_spacy` / `ner_stanza` / `ner_flair`:
```
[{'text': ..., 'label': ..., 'start': ..., 'end': ...}, ...]
```

**Checkpointing**: Results are saved every 10 articles. If the session dies, re-running skips  
already-processed articles automatically.

**Entity types**: PER, LOC, ORG, MISC (same as pipeline)


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Each notebook is self-contained; Colab wipes pip installs between sessions.

!pip install -q huggingface_hub pandas numpy tqdm
print('✅ Dependencies installed')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────

import json
import os
import pickle
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import userdata
from huggingface_hub import InferenceClient
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print('✅ Imports OK')

In [ ]:
# ── Mount Drive & load config ─────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import yaml
PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
with open(PROJECT_ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

SEED        = config.get('seed', 42)
DATA_PROC   = PROJECT_ROOT / 'Project' / 'Data' / 'Processed'
DATA_SAMP   = PROJECT_ROOT / 'Project' / 'Data' / 'Samples'
FIGURES_DIR = PROJECT_ROOT / 'Project' / 'Outputs' / 'Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Config loaded | seed={SEED}')
print(f'   DATA_PROC : {DATA_PROC}')

In [ ]:
# ── HuggingFace client setup ──────────────────────────────────────────────────
# Primary : Qwen/Qwen2.5-7B-Instruct via together provider
# Fallback : HuggingFaceH4/zephyr-7b-beta

HF_TOKEN       = userdata.get('HF_TOKEN')
PRIMARY_MODEL  = 'Qwen/Qwen2.5-7B-Instruct'
FALLBACK_MODEL = 'HuggingFaceH4/zephyr-7b-beta'

client = InferenceClient(
    provider='together',
    api_key=HF_TOKEN,
)

# Quick connectivity test
try:
    _test = client.chat_completion(
        model=PRIMARY_MODEL,
        messages=[{'role': 'user', 'content': 'Reply with the single word: OK'}],
        max_tokens=5,
    )
    print(f'✅ Primary model reachable: {PRIMARY_MODEL}')
    print(f'   Test response: {_test.choices[0].message.content.strip()}')
    ACTIVE_MODEL = PRIMARY_MODEL
except Exception as e:
    print(f'⚠️  Primary model failed ({e}), switching to fallback')
    ACTIVE_MODEL = FALLBACK_MODEL
    print(f'   Active model: {ACTIVE_MODEL}')

In [ ]:
# ── Load pipeline results ─────────────────────────────────────────────────────

PKL_PATH = DATA_PROC / 'ner_pipeline_results.pkl'
df = pd.read_pickle(PKL_PATH)

print(f'✅ Loaded pipeline results: {df.shape}')
print(f'   Columns: {df.columns.tolist()}')

In [ ]:
# ── Identify validation sample ────────────────────────────────────────────────
# Validation articles are those where Stanza/Flair were run.
# Identified by non-null ner_stanza (Stanza ran only on validation sample).

val_mask = df['ner_stanza'].notna()
df_val   = df[val_mask].copy()

print(f'✅ Validation sample: {len(df_val)} articles')

# Further split by translation quality for downstream reporting
high_q = df_val[df_val['translation_quality'] == 'high']
low_q  = df_val[df_val['translation_quality'] == 'low']
print(f'   High-quality translations : {len(high_q)}')
print(f'   Low-quality translations  : {len(low_q)}')
print(f'   Not evaluated             : {df_val["translation_quality"].eq("not_evaluated").sum()}')

In [ ]:
# ── NER prompt design ─────────────────────────────────────────────────────────
# Constraints:
#  - Labels must be exactly PER / LOC / ORG / MISC (same as pipeline)
#  - Output must be valid JSON only — no preamble, no markdown fences
#  - Keep prompt concise to minimise token usage and hallucination risk

SYSTEM_PROMPT = """You are a precise named entity recognition system.
Extract all named entities from the provided text.
Use ONLY these four labels:
  PER  – person names
  LOC  – locations, places, regions, countries
  ORG  – organisations, companies, institutions, agencies
  MISC – other named entities (products, events, laws, standards, awards, nationalities)

Rules:
- Include full entity spans (e.g. "Federal Ministry" not just "Ministry")
- Do NOT include common nouns, adjectives, or generic terms
- Do NOT include the same entity twice unless it appears with different text
- Respond with ONLY a JSON array. No explanation, no markdown.

Format:
[{"text": "<entity text>", "label": "<PER|LOC|ORG|MISC>"}, ...]

If no entities are found, respond with: []"""


def build_user_message(text: str) -> str:
    """Truncate to ~3000 words to stay within context limits."""
    words = text.split()
    if len(words) > 3000:
        text = ' '.join(words[:3000]) + ' [TRUNCATED]'
    return f"Extract named entities from this text:\n\n{text}"


print('✅ Prompt template defined')
print(f'   System prompt length: {len(SYSTEM_PROMPT)} chars')

In [ ]:
# ── Response parsing ──────────────────────────────────────────────────────────

VALID_LABELS = {'PER', 'LOC', 'ORG', 'MISC'}


def parse_llm_response(response_text: str) -> list[dict]:
    """
    Parse LLM JSON response into list of entity dicts.
    Handles: clean JSON, JSON wrapped in markdown fences, partial responses.
    Returns empty list on unrecoverable parse failure.
    """
    text = response_text.strip()

    # Strip markdown fences if present
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    text = text.strip()

    # Extract JSON array even if there's trailing text
    match = re.search(r'(\[.*?\])', text, re.DOTALL)
    if match:
        text = match.group(1)

    try:
        entities = json.loads(text)
    except json.JSONDecodeError:
        return []  # unrecoverable

    if not isinstance(entities, list):
        return []

    # Validate and normalise
    clean = []
    for ent in entities:
        if not isinstance(ent, dict):
            continue
        text_val  = str(ent.get('text', '')).strip()
        label_val = str(ent.get('label', '')).strip().upper()
        if text_val and label_val in VALID_LABELS:
            clean.append({'text': text_val, 'label': label_val})

    return clean


def add_character_offsets(entities: list[dict], source_text: str) -> list[dict]:
    """
    Add approximate start/end character offsets by searching for entity text
    in the source string. Searches case-insensitively. If an entity appears
    multiple times, records the first occurrence only (consistent with pipeline).
    Entities not found in source get start=-1, end=-1.
    """
    result = []
    source_lower = source_text.lower()

    for ent in entities:
        search_term = ent['text'].lower()
        idx = source_lower.find(search_term)
        result.append({
            'text' : ent['text'],
            'label': ent['label'],
            'start': idx,
            'end'  : idx + len(ent['text']) if idx != -1 else -1,
        })

    return result


# Smoke test
_raw = '[{"text": "Berlin", "label": "LOC"}, {"text": "BASF", "label": "ORG"}]'
_parsed = parse_llm_response(_raw)
_with_offsets = add_character_offsets(_parsed, 'The company BASF is based in Berlin.')
print('✅ Parser OK')
print(f'   Sample output: {_with_offsets}')

In [ ]:
# ── LLM call with retry logic ─────────────────────────────────────────────────

def call_llm_ner(
    text: str,
    model: str,
    max_retries: int = 3,
    retry_delay: float = 10.0,
) -> tuple[list[dict], str]:
    """
    Call the LLM for NER on `text`.
    Returns (entities, status) where status is one of:
      'ok'            – parsed successfully
      'parse_error'   – API responded but JSON was invalid
      'api_error'     – API call failed after all retries
      'no_translation'– source text was empty/NaN
    """
    if not isinstance(text, str) or not text.strip():
        return [], 'no_translation'

    for attempt in range(max_retries):
        try:
            response = client.chat_completion(
                model=model,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': build_user_message(text)},
                ],
                max_tokens=1024,
                temperature=0.0,   # deterministic — we want consistent extractions
            )
            raw = response.choices[0].message.content
            entities = parse_llm_response(raw)
            entities = add_character_offsets(entities, text)
            status   = 'ok' if entities is not None else 'parse_error'
            return entities, status

        except Exception as e:
            if attempt < max_retries - 1:
                wait = retry_delay * (attempt + 1)
                print(f'     ⚠️  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...')
                time.sleep(wait)
            else:
                print(f'     ❌ All retries exhausted: {e}')
                return [], 'api_error'


print('✅ LLM call function defined')

In [ ]:
# ── Checkpoint paths ──────────────────────────────────────────────────────────

CHECKPOINT_PATH = DATA_PROC / 'ner_llm_checkpoint.pkl'
RESULTS_PATH    = DATA_PROC / 'ner_llm_results.pkl'

def load_checkpoint() -> dict:
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, 'rb') as f:
            ckpt = pickle.load(f)
        print(f'✅ Checkpoint loaded: {len(ckpt)} articles already processed')
        return ckpt
    print('   No checkpoint found — starting fresh')
    return {}


def save_checkpoint(results: dict) -> None:
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump(results, f)


print('✅ Checkpoint paths set')
print(f'   Checkpoint : {CHECKPOINT_PATH}')
print(f'   Results    : {RESULTS_PATH}')

In [ ]:
# ── Main inference loop ───────────────────────────────────────────────────────
# Rate limit: ~1 req/s to stay well within free tier limits.
# Checkpoints every 10 articles.
# Skips already-processed article_ids automatically.

RATE_LIMIT_SLEEP = 1.5   # seconds between API calls
CHECKPOINT_EVERY = 10    # save to Drive every N articles

results = load_checkpoint()  # {article_id: {'entities': [...], 'status': '...'}}

# Identify articles still to process
todo = df_val[~df_val['article_id'].isin(results.keys())]
print(f'\n🚀 Starting inference')
print(f'   Total validation articles : {len(df_val)}')
print(f'   Already processed         : {len(results)}')
print(f'   Remaining                 : {len(todo)}')
print(f'   Model                     : {ACTIVE_MODEL}')
print(f'   Estimated time            : ~{len(todo) * RATE_LIMIT_SLEEP / 60:.1f} min at {RATE_LIMIT_SLEEP}s/req\n')

status_counts = {'ok': 0, 'parse_error': 0, 'api_error': 0, 'no_translation': 0}

for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo), desc='LLM NER')):
    article_id   = row['article_id']
    content_en   = row.get('content_en', '')

    entities, status = call_llm_ner(content_en, model=ACTIVE_MODEL)
    status_counts[status] += 1

    results[article_id] = {
        'entities'          : entities,
        'entity_count'      : len(entities),
        'status'            : status,
        'translation_quality': row.get('translation_quality', 'unknown'),
        'model'             : ACTIVE_MODEL,
    }

    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(results)
        tqdm.write(f'   💾 Checkpoint saved ({len(results)} articles processed)')

    time.sleep(RATE_LIMIT_SLEEP)

# Final save
save_checkpoint(results)

print(f'\n✅ Inference complete')
print(f'   Status breakdown: {status_counts}')

In [ ]:
# ── Merge LLM results back into DataFrame ─────────────────────────────────────
# Adds ner_llm and llm_entity_count columns (same schema as pipeline columns)
# Articles not in validation sample get None.

def get_llm_entities(article_id):
    r = results.get(article_id)
    return r['entities'] if r else None

def get_llm_count(article_id):
    r = results.get(article_id)
    return r['entity_count'] if r else None

def get_llm_status(article_id):
    r = results.get(article_id)
    return r['status'] if r else None

df['ner_llm']        = df['article_id'].map(get_llm_entities)
df['llm_entity_count'] = df['article_id'].map(get_llm_count)
df['llm_status']     = df['article_id'].map(get_llm_status)

# Validation sample only view
df_val_merged = df[df['ner_stanza'].notna()].copy()

print(f'✅ LLM results merged into DataFrame')
print(f'   Validation articles with LLM results: {df_val_merged["ner_llm"].notna().sum()}')
print(f'   Mean LLM entity count: {df_val_merged["llm_entity_count"].mean():.1f}')
print(f'   Status breakdown:')
print(df_val_merged['llm_status'].value_counts().to_string())

In [ ]:
# ── Entity count comparison: pipeline vs LLM ──────────────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Entity Count Distribution: Pipeline Systems vs LLM', fontsize=13, fontweight='bold')

# Left: distributions
ax = axes[0]
bins = range(0, 120, 5)
ax.hist(df_val_merged['spacy_entity_count'].dropna(),  bins=bins, alpha=0.6, label='spaCy (DE)',   color='#2196F3')
ax.hist(df_val_merged['stanza_entity_count'].dropna(), bins=bins, alpha=0.6, label='Stanza (DE)',  color='#4CAF50')
ax.hist(df_val_merged['flair_entity_count'].dropna(),  bins=bins, alpha=0.6, label='Flair (DE)',   color='#FF9800')
ax.hist(df_val_merged['llm_entity_count'].dropna(),    bins=bins, alpha=0.6, label='LLM / Qwen (EN)', color='#9C27B0')
ax.set_xlabel('Entities per article')
ax.set_ylabel('Article count')
ax.set_title('Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Right: mean counts with error bars
ax2 = axes[1]
systems  = ['spaCy\n(DE)', 'Stanza\n(DE)', 'Flair\n(DE)', 'Qwen\n(EN)']
cols     = ['spacy_entity_count', 'stanza_entity_count', 'flair_entity_count', 'llm_entity_count']
colors   = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
means    = [df_val_merged[c].mean() for c in cols]
stds     = [df_val_merged[c].std()  for c in cols]

bars = ax2.bar(systems, means, color=colors, alpha=0.8, edgecolor='white')
ax2.errorbar(systems, means, yerr=stds, fmt='none', color='black', capsize=5, linewidth=1.5)
for bar, mean in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{mean:.1f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Mean entity count')
ax2.set_title('Mean ± SD per system')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = FIGURES_DIR / 'ner_llm_entity_counts.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure saved: {fig_path}')

In [ ]:
# ── Label distribution per system ─────────────────────────────────────────────

def extract_label_counts(ner_col: pd.Series, label_set=('PER', 'LOC', 'ORG', 'MISC')) -> dict:
    """Count total entities per label across all articles in the series."""
    counts = {lbl: 0 for lbl in label_set}
    for ents in ner_col.dropna():
        if isinstance(ents, list):
            for e in ents:
                lbl = e.get('label', '').upper()
                if lbl in counts:
                    counts[lbl] += 1
    return counts


label_data = {
    'spaCy (DE)' : extract_label_counts(df_val_merged['ner_spacy']),
    'Stanza (DE)': extract_label_counts(df_val_merged['ner_stanza']),
    'Flair (DE)' : extract_label_counts(df_val_merged['ner_flair']),
    'Qwen (EN)'  : extract_label_counts(df_val_merged['ner_llm']),
}

label_df = pd.DataFrame(label_data).T
print('Entity label distribution across validation sample:')
print(label_df.to_string())

# Normalise to proportions
label_prop = label_df.div(label_df.sum(axis=1), axis=0)
print('\nProportions:')
print(label_prop.round(3).to_string())

In [ ]:
# ── Label distribution plot ───────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(label_df.columns))
width   = 0.2
systems = list(label_data.keys())
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, (sys_name, color) in enumerate(zip(systems, colors)):
    vals = [label_df.loc[sys_name, lbl] for lbl in label_df.columns]
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, vals, width, label=sys_name, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(label_df.columns, fontsize=12)
ax.set_ylabel('Total entity count (validation sample)')
ax.set_title('Named Entity Label Distribution by System', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path2 = FIGURES_DIR / 'ner_llm_label_distribution.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure saved: {fig_path2}')

In [ ]:
# ── Translation quality effect on LLM entity count ───────────────────────────
# Key question: does lower BERTScore translation quality reduce LLM entity extraction?

from scipy import stats

high_counts = df_val_merged[df_val_merged['translation_quality'] == 'high']['llm_entity_count'].dropna()
low_counts  = df_val_merged[df_val_merged['translation_quality'] == 'low']['llm_entity_count'].dropna()

print('LLM entity count by translation quality:')
print(f'  High quality (BERTScore ≥0.75): n={len(high_counts)}, mean={high_counts.mean():.1f}, sd={high_counts.std():.1f}')
print(f'  Low quality  (BERTScore <0.75): n={len(low_counts)},  mean={low_counts.mean():.1f}, sd={low_counts.std():.1f}')

if len(high_counts) > 0 and len(low_counts) > 0:
    stat, pval = stats.mannwhitneyu(high_counts, low_counts, alternative='two-sided')
    print(f'\nMann-Whitney U test: U={stat:.1f}, p={pval:.4f}')
    print(f'  {"Significant" if pval < 0.05 else "Not significant"} difference (α=0.05)')

# Correlation with BERTScore
valid = df_val_merged[['bertscore_f1', 'llm_entity_count']].dropna()
if len(valid) > 2:
    r, p = stats.pearsonr(valid['bertscore_f1'], valid['llm_entity_count'])
    print(f'\nPearson r (BERTScore vs LLM entity count): r={r:.3f}, p={p:.4f}')

In [ ]:
# ── Sample entity inspection ──────────────────────────────────────────────────
# Spot-check 5 articles to verify LLM output quality qualitatively

sample_ids = df_val_merged[df_val_merged['llm_status'] == 'ok'].sample(5, random_state=SEED)['article_id'].tolist()

for art_id in sample_ids:
    row = df_val_merged[df_val_merged['article_id'] == art_id].iloc[0]
    llm_ents   = row['ner_llm'] or []
    spacy_ents = row['ner_spacy'] or []

    print(f'\n── {art_id} | {row["source"]} | {row["translation_quality"]} translation')
    print(f'   Title: {str(row["title"])[:60]}')
    print(f'   spaCy  ({len(spacy_ents)} ents): {[(e["text"], e["label"]) for e in spacy_ents[:5]]}')
    print(f'   LLM    ({len(llm_ents)} ents): {[(e["text"], e["label"]) for e in llm_ents[:5]]}')

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────

summary = {
    'model'                      : ACTIVE_MODEL,
    'validation_articles'         : len(df_val_merged),
    'successful_extractions'      : int((df_val_merged['llm_status'] == 'ok').sum()),
    'parse_errors'                : int((df_val_merged['llm_status'] == 'parse_error').sum()),
    'api_errors'                  : int((df_val_merged['llm_status'] == 'api_error').sum()),
    'no_translation'              : int((df_val_merged['llm_status'] == 'no_translation').sum()),
    'mean_llm_entities'           : round(float(df_val_merged['llm_entity_count'].mean()), 2),
    'mean_spacy_entities'         : round(float(df_val_merged['spacy_entity_count'].mean()), 2),
    'mean_stanza_entities'        : round(float(df_val_merged['stanza_entity_count'].mean()), 2),
    'mean_flair_entities'         : round(float(df_val_merged['flair_entity_count'].mean()), 2),
    'pipeline_fleiss_kappa'       : 0.717,   # from notebook 03
    'label_counts_llm'            : extract_label_counts(df_val_merged['ner_llm']),
}

summary_path = DATA_PROC / 'ner_llm_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('✅ Summary:')
for k, v in summary.items():
    print(f'   {k}: {v}')

In [ ]:
# ── Save final results ────────────────────────────────────────────────────────
# Save the full DataFrame (all 1115 rows) with new LLM columns.
# Non-validation articles will have None for LLM columns.

df.to_pickle(RESULTS_PATH)
print(f'✅ Results saved: {RESULTS_PATH}')
print(f'   Shape: {df.shape}')
print(f'   New columns: ner_llm, llm_entity_count, llm_status')

# Clean up checkpoint (optional — keep it in case needed)
print(f'\n   Checkpoint retained at: {CHECKPOINT_PATH}')
print('   Delete manually if no longer needed.')

## Notebook summary

| Item | Detail |
|------|--------|
| Model | `Qwen/Qwen2.5-7B-Instruct` via `together` provider |
| Input | English translations (`content_en`) of 199-article validation sample |
| Entity types | PER, LOC, ORG, MISC |
| Temperature | 0.0 (deterministic) |
| Output schema | `[{'text', 'label', 'start', 'end'}]` — same as pipeline columns |
| Saved files | `ner_llm_results.pkl`, `ner_llm_summary.json`, `ner_llm_checkpoint.pkl` |
| Figures | `ner_llm_entity_counts.png`, `ner_llm_label_distribution.png` |

**Next**: `05_ner_comparison.ipynb` — head-to-head F1, speed, and transparency trade-offs.
